In [1]:
her2st_folder = '../data/her2st/data'
stnet_folder = '../data/ST/BC'
skin_folder = '../data/skin'
pcw_folder = '../data/ST-images/PCW'
mouse_folder = '../data/ST-images/MOUSE'

her2st_imgs_folder = her2st_folder + '/ST-imgs'
her2st_st_postion = her2st_folder + '/ST-spotfiles'
her2st_gene_expression = her2st_folder + '/ST-cnts'
print(her2st_imgs_folder)
print(her2st_st_postion)
print(her2st_gene_expression)

stnet_imgs_folder = stnet_folder + '/ST-imgs'
stnet_st_postion = stnet_folder + '/ST-spotfiles'
stnet_gene_expression = stnet_folder + '/ST-cnts'
print(stnet_imgs_folder)
print(stnet_st_postion)
print(stnet_gene_expression)

skin_imgs_folder = skin_folder + '/ST-imgs'
skin_st_postion = skin_folder + '/ST-spotfiles'
skin_gene_expression = skin_folder + '/ST-cnts'
print(skin_imgs_folder)
print(skin_st_postion)
print(skin_gene_expression)

pcw_imgs_folder =  pcw_folder + '/ST-imgs'
pcw_st_postion = pcw_folder + '/ST-spotfiles'
pcw_gene_expression = pcw_folder + '/ST-cnts'
print(pcw_imgs_folder)
print(pcw_st_postion)
print(pcw_gene_expression)

mouse_imgs_folder =  mouse_folder + '/ST-imgs'
mouse_st_postion = mouse_folder + '/ST-spotfiles'
mouse_gene_expression = mouse_folder + '/ST-cnts'
print(mouse_imgs_folder)
print(mouse_st_postion)
print(mouse_gene_expression)

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import json

registration_xfeat_her2st_dir = []

registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/A/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/B/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/C/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/D/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/E/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/F/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/G/registration_xfeat')
registration_xfeat_her2st_dir.append(her2st_imgs_folder + '/H/registration_xfeat')

Image.MAX_IMAGE_PIXELS = None 

import ast
def parse_np_int64_list(value):
    if pd.isna(value) or value == '[]':  
        return []
    value = value.replace("np.int64(", "").replace(")", "")
    return [int(x) for x in ast.literal_eval(value)]

def build_dataset(sample_dir, spot_mapping_file, position_file,gene_expression_dir,image_files,sample_name,output_dir):
    print(sample_dir)
    print(spot_mapping_file)
    print(position_file)
    print(gene_expression_dir)
    print(output_dir)
    print(image_files)
    spot_mapping = pd.read_csv(spot_mapping_file)
    column_names = spot_mapping.columns.tolist()
    print("All Column Names:")
    print(column_names)
    all_layer_data = {}
    for index, row in spot_mapping.iterrows():
        print(row)
        sample_idx = 0
        layer_data = {}
        for i in row :
            i = parse_np_int64_list(i)
            print(i)
            sample_spot = column_names[sample_idx]
            sample = sample_spot.split('_')[0]
            print(sample)
            selction_path = os.path.join(position_file,sample+'_selection.tsv')
            print(selction_path)
            selction_data = pd.read_csv(selction_path, sep='\t')
            image_path = os.path.join(image_files,sample,sample+'.jpg')
            print(image_path)
            gene_path = os.path.join(gene_expression_dir,sample+'.tsv')
            print(gene_path)
            gene_data = pd.read_csv(gene_path, sep='\t',index_col=0)
            layer_name = f"layer_{sample_idx}"
            layer_data[layer_name] = {
                'gene_expressions': [],
                'cropped_image_names': []
            }
            for j in i:
                data = selction_data.iloc[j]
                pixel_x = data['pixel_x']
                pixel_y = data['pixel_y']
                x = int(data['x'])
                y = int(data['y'])
                cropped_image_name = f'{sample}_{x}x{y}.jpg'
                cropped_image_path = os.path.join(output_dir, cropped_image_name)
                layer_data[layer_name]['cropped_image_names'].append(cropped_image_name)
                spot_key = f"{x}x{y}"
                if spot_key in gene_data.index:
                    gene_expression = gene_data.loc[spot_key].tolist()
                    print(f"Gene Expression for {spot_key}: {gene_expression}")
                else:
                    print(f"Spot Index: {spot_key} not found in gene_data")
                    gene_expression = []
                layer_data[layer_name]['gene_expressions'].append(gene_expression)
            
            sample_idx+=1
        print(f"Row {index} Layer Data:")
        for layer_name, data in layer_data.items():
            print(f"Layer: {layer_name}")
            print(f"Cropped Image Names: {data['cropped_image_names']}")
            print(f"Gene Expressions: {data['gene_expressions']}")

        all_layer_data[f"row_{index}"] = layer_data
    all_layer_data_file = os.path.join(output_dir, f'{sample_name}_all_layer_data.json')
    with open(all_layer_data_file, 'w') as f:
        json.dump(all_layer_data, f, indent=4)
    print(f"All layer data saved to: {all_layer_data_file}")
    all_layer_data_file_npy = os.path.join(output_dir, f'{sample_name}_all_layer_data.npy')
    np.save(all_layer_data_file_npy, all_layer_data)
    print(f"All layer data saved to: {all_layer_data_file_npy}")
     
for sample_dir in registration_xfeat_her2st_dir:
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    spot_mapping_file = os.path.join(sample_dir, f'{sample_name}_spot_mapping.csv')
    potion_file = os.path.join(her2st_st_postion)
    output_dir = os.path.join('./', 'dataset')
    os.makedirs(output_dir, exist_ok=True)
    gene_expression_dir = './her2st_top250_heg'
    image_files = os.path.join(her2st_imgs_folder,sample_name)
    print(image_files)
    build_dataset(sample_dir, spot_mapping_file, potion_file,gene_expression_dir,image_files,sample_name,output_dir)
    

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import json

registration_xfeat_her2st_dir = []
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/A/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/B/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/C/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/D/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/E/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/F/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/G/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/H/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/I/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/J/registration_xfeat')
registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/K/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/L/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/M/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/N/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/O/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/P/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/Q/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/R/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/S/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/T/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/U/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/V/registration_xfeat')
# registration_xfeat_her2st_dir.append(stnet_imgs_folder + '/W/registration_xfeat')

Image.MAX_IMAGE_PIXELS = None

import ast
def parse_np_int64_list(value):
    if pd.isna(value) or value == '[]':  
        return []
    value = value.replace("np.int64(", "").replace(")", "")
    return [int(x) for x in ast.literal_eval(value)]

def build_dataset(sample_dir, spot_mapping_file, position_file,gene_expression_dir,image_files,sample_name,output_dir):
    print(sample_dir)
    print(spot_mapping_file)
    print(position_file)
    print(gene_expression_dir)
    print(output_dir)
    print(image_files)
    spot_mapping = pd.read_csv(spot_mapping_file)
    column_names = spot_mapping.columns.tolist()
    print("All Column Names:")
    print(column_names)
    all_layer_data = {}
    for index, row in spot_mapping.iterrows():
        print(row)
        sample_idx = 0
        layer_data = {}
        for i in row :
            i = parse_np_int64_list(i)
            print(i)
            sample_spot = column_names[sample_idx]
            sample = sample_spot.split('_')[0]
            print(sample)
            selction_path = os.path.join(position_file,sample+'.csv')
            print(selction_path)
            selction_data = pd.read_csv(selction_path)
            image_path = os.path.join(image_files,sample,sample+'.jpg')
            print(image_path)
            gene_path = os.path.join(gene_expression_dir,sample+'.tsv')
            print(gene_path)
            gene_data = pd.read_csv(gene_path, sep='\t',index_col=0)
            layer_name = f"layer_{sample_idx}"
            layer_data[layer_name] = {
                'gene_expressions': [],
                'cropped_image_names': []
            }
            for j in i:
                data = selction_data.iloc[j]
                pixel_x = data['pixel_x']
                pixel_y = data['pixel_y']
                x = int(data['x'])
                y = int(data['y'])
                cropped_image_name = f'{sample}_{x}x{y}.jpg'
                cropped_image_path = os.path.join(output_dir, cropped_image_name)
                layer_data[layer_name]['cropped_image_names'].append(cropped_image_name)
                spot_key = f"{x}x{y}"
                if spot_key in gene_data.index:
                    gene_expression = gene_data.loc[spot_key].tolist()
                    print(f"Gene Expression for {spot_key}: {gene_expression}")
                else:
                    print(f"Spot Index: {spot_key} not found in gene_data")
                    gene_expression = []
                layer_data[layer_name]['gene_expressions'].append(gene_expression)
            sample_idx+=1
        print(f"Row {index} Layer Data:")
        for layer_name, data in layer_data.items():
            print(f"Layer: {layer_name}")
            print(f"Cropped Image Names: {data['cropped_image_names']}")
            print(f"Gene Expressions: {data['gene_expressions']}")

        all_layer_data[f"row_{index}"] = layer_data
    all_layer_data_file = os.path.join(output_dir, f'{sample_name}_all_layer_data.json')
    with open(all_layer_data_file, 'w') as f:
        json.dump(all_layer_data, f, indent=4)
    print(f"All layer data saved to: {all_layer_data_file}")
    all_layer_data_file_npy = os.path.join(output_dir, f'{sample_name}_all_layer_data.npy')
    np.save(all_layer_data_file_npy, all_layer_data)
    print(f"All layer data saved to: {all_layer_data_file_npy}")



for sample_dir in registration_xfeat_her2st_dir:
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    spot_mapping_file = os.path.join('./stnet_mappings', f'{sample_name}_spot_mapping.csv')
    potion_file = os.path.join(stnet_st_postion)
    output_dir = os.path.join('./', 'stnet_dataset_normal_smooth')
    os.makedirs(output_dir, exist_ok=True)
    gene_expression_dir = './stnet_heg250_normal_smooth'
    image_files = os.path.join(stnet_imgs_folder,sample_name)
    print(image_files)
    build_dataset(sample_dir, spot_mapping_file, potion_file,gene_expression_dir,image_files,sample_name,output_dir)
    

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import json

registration_xfeat_her2st_dir = []
registration_xfeat_her2st_dir.append(skin_imgs_folder + '/A/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/B/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/C/registration_xfeat')
# registration_xfeat_her2st_dir.append(skin_imgs_folder + '/D/registration_xfeat')

Image.MAX_IMAGE_PIXELS = None 

import ast
def parse_np_int64_list(value):
    if pd.isna(value) or value == '[]':  
        return []
    value = value.replace("np.int64(", "").replace(")", "")
    return [int(x) for x in ast.literal_eval(value)]

def build_dataset(sample_dir, spot_mapping_file, position_file,gene_expression_dir,image_files,sample_name,output_dir):
    print(sample_dir)
    print(spot_mapping_file)
    print(position_file)
    print(gene_expression_dir)
    print(output_dir)
    print(image_files)
    spot_mapping = pd.read_csv(spot_mapping_file)
    column_names = spot_mapping.columns.tolist()
    print("All Column Names:")
    print(column_names)
    all_layer_data = {}
    for index, row in spot_mapping.iterrows():
        print(row)
        sample_idx = 0
        layer_data = {}
        for i in row :
            i = parse_np_int64_list(i)
            print(i)
            sample_spot = column_names[sample_idx]
            sample = sample_spot.split('_')[0]
            print(sample)
            selction_path = os.path.join(position_file,sample+'.tsv')
            print(selction_path)
            selction_data = pd.read_csv(selction_path, sep='\t')
            image_path = os.path.join(image_files,sample,sample+'.jpg')
            print(image_path)
            gene_path = os.path.join(gene_expression_dir,sample+'.tsv')
            print(gene_path)
            gene_data = pd.read_csv(gene_path, sep='\t',index_col=0)
            layer_name = f"layer_{sample_idx}"
            layer_data[layer_name] = {
                'gene_expressions': [],
                'cropped_image_names': []
            }
            for j in i:
                data = selction_data.iloc[j]
                pixel_x = data['pixel_x']
                pixel_y = data['pixel_y']
                x = int(data['x'])
                y = int(data['y'])
                cropped_image_name = f'{sample}_{x}x{y}.jpg'
                cropped_image_path = os.path.join(output_dir, cropped_image_name)
                layer_data[layer_name]['cropped_image_names'].append(cropped_image_name)
                spot_key = f"{x}x{y}"
                if spot_key in gene_data.index:
                    gene_expression = gene_data.loc[spot_key].tolist()
                    print(f"Gene Expression for {spot_key}: {gene_expression}")
                else:
                    print(f"Spot Index: {spot_key} not found in gene_data")
                    gene_expression = []
                layer_data[layer_name]['gene_expressions'].append(gene_expression)
            
            sample_idx+=1
        print(f"Row {index} Layer Data:")
        for layer_name, data in layer_data.items():
            print(f"Layer: {layer_name}")
            print(f"Cropped Image Names: {data['cropped_image_names']}")
            print(f"Gene Expressions: {data['gene_expressions']}")

        all_layer_data[f"row_{index}"] = layer_data
    all_layer_data_file = os.path.join(output_dir, f'{sample_name}_all_layer_data.json')
    with open(all_layer_data_file, 'w') as f:
        json.dump(all_layer_data, f, indent=4)
    print(f"All layer data saved to: {all_layer_data_file}")
    all_layer_data_file_npy = os.path.join(output_dir, f'{sample_name}_all_layer_data.npy')
    np.save(all_layer_data_file_npy, all_layer_data)
    print(f"All layer data saved to: {all_layer_data_file_npy}")



for sample_dir in registration_xfeat_her2st_dir:
    sample_name = os.path.basename(os.path.dirname(sample_dir))
    spot_mapping_file = os.path.join('./skin_mappings', f'{sample_name}_spot_mapping.csv')
    potion_file = os.path.join(skin_st_postion)
    output_dir = os.path.join('./', 'skin_dataset_normal_smooth_a3')
    os.makedirs(output_dir, exist_ok=True)
    gene_expression_dir = './skin_heg250_normal_smooth'
    image_files = os.path.join(skin_imgs_folder,sample_name)
    print(image_files)
    build_dataset(sample_dir, spot_mapping_file, potion_file,gene_expression_dir,image_files,sample_name,output_dir)
    